# Verification

Inspect a saved processing history, reproduce its results, and export the workflow
and recorded environment for reuse.

## Load a recorded result

Complete [DDA-TMT](dda-tmt.ipynb) with the current `msmu` version, then run this
notebook from the same working directory. Its processed result is
`dda_tmt_PXD013361.h5mu`.
You can replace this path with your own recorded analysis.


In [1]:
import logging
import runpy

import msmu as mm

logging.getLogger("msmu").setLevel(logging.WARNING)

mdata = mm.read_h5mu("dda_tmt_PXD013361.h5mu")
reference_hash = mm.pv.compute_hash(mdata)
original_log = mm.pv.get_log(mdata)

For verified replay, the history must start from recorded file readers,
contain hashes for supported operations that return a `MuData`, and have no gaps
from unrecorded changes. Keep the original input files or URLs available.
See the [replay API](../reference/pv/replay.md) for supported histories, arguments,
and restrictions.
PCA and UMAP require a recorded integer `random_state` (PCA defaults to `0`).

## Inspect the recorded calls

[`mm.pv.get_log()`](../reference/pv/get_log.md) returns a decoded copy of the history.
Inspect the last event's function, parameters, environment reference, and parents:


In [2]:
log = mm.pv.get_log(mdata)
event = log["events"][-1]
print(event["function_path"], event["parameters"])
print(event["environment_id"], event["parents"])

msmu._preprocessing._filter.apply_filter {'mdata': {'id': 'd4270e67-5cf8-4310-8a4a-c9128ec5b8c8', 'role': 'arguments/mdata', 'type': 'MuData', 'hash': {'status': 'completed', 'algorithm': 'sha256', 'value': '12cf46406a98663221f6b54566ab0466f61916d49c988146098513b04b314500', 'normalization': 'msmu-v2', 'duration_seconds': 0.01957383399712853}, 'source_event': '10f2dc23-33a5-42a8-9828-5136ac0754a4'}, 'modality': 'protein', 'on': 'var', 'columns': None}
020aec2375bb290ec81dcd2df6ad3f1fb7795872aba9b7bd94790f9760918def ['10f2dc23-33a5-42a8-9828-5136ac0754a4']


`mm.pv.replay()` and `mm.pv.to_script()` use these events to reconstruct supported
workflows.

## Export and prepare the recorded environment

[`mm.pv.to_env()`](../reference/pv/to_env.md) exports recorded Python and package
versions without changing the data or installing packages:


In [3]:
mm.pv.to_env(mdata, "requirements.txt", format="uv")
mm.pv.to_env(mdata, "environment.yml", format="conda")

An existing destination file is overwritten. Omit the filename to return the
specification as text. If the history contains different installation environments,
inspect their IDs and explicitly select one of the printed values:


```python
print(log["environments"].keys())
text = mm.pv.to_env(log, environment_id="<recorded ID>", format="uv")
```

For `uv`, use the Python version printed in the exported file:

```bash
# Replace 3.12.8 with the recorded version printed in requirements.txt.
uv venv --python 3.12.8 .venv-replay
uv pip sync --python .venv-replay requirements.txt
```

For `conda`:

```bash
conda env create --name msmu-replay --file environment.yml
conda activate msmu-replay
```

These files pin Python and package versions; they are not complete environment
lockfiles. Package sources, native libraries, platform differences, and uncommitted
source changes need separate handling. Use the prepared environment and matching
`msmu` version for the replay or script below.

## Replay the workflow

[`mm.pv.replay()`](../reference/pv/replay.md) reruns the recorded calls from their
original inputs and returns a new `MuData` with freshly recorded history:


In [4]:
replayed = mm.pv.replay(mdata)

WARNING - apply_sdrf_to_obs: psm columns not projectable under 'comment[label]'; kept only in uns['sdrf']: ['assay name', 'comment[data file]', 'comment[fraction identifier]']


WARNING - Purity column not found in psm modality for TMT data. Skipping purity filtering.


The supplied object is unchanged. With the default `verify=True`, replay uses the
recorded hashing policy, checks input/output hashes, and stops at the first
mismatch. Environment differences produce warnings; replay does not install
packages.

If inputs have moved, the `sources` argument maps their exact recorded paths or
URLs to replacements. This changes locations, not expected hashes; source locations
stored as metadata can also affect output hashes. See the
[replay API](../reference/pv/replay.md) before remapping inputs.

To rerun without hash verification:


In [5]:
unverified = mm.pv.replay(mdata, verify=False)

WARNING - apply_sdrf_to_obs: psm columns not projectable under 'comment[label]'; kept only in uns['sdrf']: ['assay name', 'comment[data file]', 'comment[fraction identifier]']


WARNING - Purity column not found in psm modality for TMT data. Skipping purity filtering.


This also skips new hash calculation. Unsupported histories and known gaps are
still rejected, and the caller's hashing setting is restored afterward, including
on failure.

## Generate a verified script

[`mm.pv.to_script()`](../reference/pv/to_script.md) returns Python source or writes
it to a file without reading the inputs or running the analysis:


In [6]:
script = mm.pv.to_script(mdata, verify=True)
preview = [line for line in script.splitlines() if line.startswith("# mm.")]
print("\n".join(preview[:5]))

mm.pv.to_script(mdata, filename="workflow.py", verify=True)

# mm.read_sage
# mm.pp.attach_sdrf
# mm.pp.apply_sdrf_to_obs
# mm.pp.add_filter
# mm.pp.apply_filter


An existing destination is overwritten after validation. When given a `MuData`,
verified export first checks its current content against the last recorded output;
a mismatch leaves the destination unchanged. Passing a decoded log cannot check
the current data.

Run `workflow.py` in the prepared environment with its original inputs available.
The script checks the environment and recorded hashes using the recorded hashing
policy, then leaves the result in `mdata`. The same supported-workflow restrictions
as replay apply.


In [7]:
script_result = runpy.run_path("workflow.py", run_name="__main__")["mdata"]

WARNING - apply_sdrf_to_obs: psm columns not projectable under 'comment[label]'; kept only in uns['sdrf']: ['assay name', 'comment[data file]', 'comment[fraction identifier]']


WARNING - Purity column not found in psm modality for TMT data. Skipping purity filtering.


## Customize the workflow

Generate a script without comparisons against the original hashes, then edit its
input paths, parameters, or processing calls:



In [8]:
mm.pv.to_script(mdata, filename="modified_workflow.py", verify=False)

Unlike direct replay with `verify=False`, generated scripts enable hashing and
record a new history even when verification is disabled. History validation and
environment checks still apply.


## Check the results

Confirm that verified replay and the generated script reproduce the saved result,
and that the original data and history remain unchanged.

In [9]:
assert mm.pv.compute_hash(replayed) == reference_hash
assert mm.pv.compute_hash(script_result) == reference_hash
assert mm.pv.compute_hash(mdata) == reference_hash
assert mm.pv.get_log(mdata) == original_log
print("Replay and verified script match the saved result.")
print("Original data and history are unchanged.")
print("Hash:", reference_hash)

Replay and verified script match the saved result.
Original data and history are unchanged.
Hash: 83fffab334f177f7fb296af2fb436bbe8185825bd5fbcd9d95890600673eaba1
